# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset metadata is available via a Croissant schema URL for reproducible loading and processing.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using mlcroissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Metadata Loaded.")
print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We iterate through the dataset to display all Record Sets with their `@id`, name, and included Fields with their corresponding `@id`.

In [ ]:
# List all record sets and fields with their @id
record_sets = []
print("Available Record Sets and Fields:\n")

for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}  |  name: {field.name}")
    print()
    record_sets.append(rs.id)

# Choose a sample record set for further processing
if len(record_sets) == 0:
    print("No record sets found in metadata.")
else:
    print(f"Record set IDs: {record_sets}")

## 3. Data Extraction
Let's load the data from a specific record set into a DataFrame for analysis. All access is done via the record set and field `@id`s reported above.

In [ ]:
# Extract data from each record set into pandas DataFrames
# (Replace '<record_set_id>' with an actual @id from the previous step)

# We collect DataFrames in a dictionary for easy access
dataframes = {}

for record_set_id in record_sets:
    try:
        # Load all records for the record set
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"[Warning] No records found for RecordSet: {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Failed loading records for {record_set_id}: {e}")

# Select the main record set to use in the next steps
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing '{main_record_set_id}' for example analysis.")
    main_df = dataframes[main_record_set_id]
else:
    # In case there are no DataFrames loaded
    main_record_set_id = None
    main_df = None

## 4. Exploratory Data Analysis (EDA)
Apply common steps—filter records, normalize numeric fields, and group data by categorical fields.

Replace variables with appropriate `@id` column names from the previous step's DataFrame output.

In [ ]:
import numpy as np

# Example: Assume we have a numeric field 'cr:peptide_count' and group by a field 'cr:description'.
# Adapt these field names as per your specific dataset overview above.
if main_df is not None:
    # Try to pick the first numeric column for demonstration
    numeric_candidates = main_df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric candidates: {numeric_candidates}")
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]   # Example: '@id' for the numeric field
        threshold = main_df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} found.")

        # Normalize the field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}. Top records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Pick a string column to group by
        group_field_candidates = main_df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        if len(group_field_candidates) > 0:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No string fields to group by.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No DataFrame available for analysis.")

## 5. Visualization
Visualize the distribution of a selected numeric field, or relationships between fields. Use matplotlib or seaborn for common plots.

In [ ]:
# Visualize a numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and 'numeric_field' in locals() and numeric_field in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
We have:
- Loaded the FAIR² mass spectrometry dataset from a Croissant schema URL using `mlcroissant`
- Explored the record sets, fields, and their IDs
- Extracted and previewed tabular data from the record set
- Applied basic EDA: filtering, normalization, grouping
- Produced a simple visualization of field distributions

For deeper domain-specific analysis (e.g., protein identification statistics, peptide sequence examination) refer to the original dataset documentation and experiment with additional fields and custom visualizations.